# Lekcja 10 - Agenci AI w produkcji

W tej lekcji poznasz **wzorce produkcyjne** dla agentów AI z użyciem Microsoft Agent Framework (Python). Omawiamy:

- **Obserwowalność** — dodawanie pomiarów czasu i logowania do interakcji agenta
- **Ewaluacja** — użycie agenta oceniającego do oceniania jakości odpowiedzi
- **Zarządzanie kosztami** — strategie optymalizacji tokenów i wyboru modelu

Scenariusz to **agent podróży**, który pomaga użytkownikom planować podróże, z monitorowaniem i ewaluacją nałożonymi na to.


## Konfiguracja


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Rozważania dotyczące wdrożenia

Przeniesienie agentów AI z prototypów do środowiska produkcyjnego wymaga uważnego zwrócenia uwagi na trzy filary:

1. **Observability** — Musisz mieć wgląd w to, co robi agent, ile to trwa i których narzędzi używa. Bez śledzenia i logowania debugowanie problemów w środowisku produkcyjnym jest praktycznie niemożliwe.

2. **Evaluation** — Zautomatyzowane kontrole jakości zapewniają, że odpowiedzi agenta pozostają dokładne, kompletne i pomocne w czasie. Agent-ewaluator może oceniać odpowiedzi według zdefiniowanych kryteriów.

3. **Cost Management** — Wykorzystanie tokenów bezpośrednio wpływa na koszty. Strategie takie jak optymalizacja promptów, dobór modelu i buforowanie pomagają utrzymać wydatki pod kontrolą bez poświęcania jakości.


## Tworzenie obserwowalnego agenta

Definiujemy narzędzia podróżne i opakowujemy wywołanie agenta pomiarem czasu, aby móc monitorować opóźnienia. W środowisku produkcyjnym należałoby to zintegrować z OpenTelemetry lub podobnym backendem śledzenia.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Wzorce ewaluacji

Powszechny wzorzec produkcyjny polega na użyciu drugiego agenta jako **ewaluatora**. Ewaluator ocenia odpowiedź głównego agenta według z góry określonych kryteriów, takich jak kompletność, dokładność i przydatność.

To umożliwia:
- Zautomatyzowane bramki jakości przed udostępnieniem odpowiedzi użytkownikom
- Wykrywanie regresji, gdy zmieniają się prompty lub modele
- Ciągłe monitorowanie wydajności agenta w czasie


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie zarządzania kosztami

Kontrolowanie kosztów jest kluczowe dla produkcyjnych agentów AI. Oto kluczowe strategie:

| Strategia | Opis |
|---|---|
| **Optymalizacja promptu** | Utrzymuj instrukcje systemowe zwięzłe. Usuń zbędny kontekst, aby zmniejszyć liczbę tokenów wejściowych. |
| **Wybór modelu** | Używaj mniejszych, tańszych modeli (np. GPT-4o-mini) do prostych zadań takich jak klasyfikacja czy ekstrakcja, a większe modele rezerwuj do złożonego wnioskowania. |
| **Buforowanie** | Buforuj wyniki narzędzi i częste zapytania, aby uniknąć zbędnych wywołań API. |
| **Limity tokenów** | Ustaw `max_tokens` limity, aby zapobiec nieoczekiwanie długim odpowiedziom. |
| **Grupowanie** | Grupuj wiele zapytań użytkowników w jedno wywołanie API, gdy to możliwe. |

W praktyce dobrze sprawdza się podejście warstwowe: kieruj proste żądania do szybkiego, taniego modelu, a tylko złożone zapytania eskaluj do bardziej zaawansowanego.


## Podsumowanie

W tej lekcji dowiedziałeś się, jak:

1. **Dodać obserwowalność** do interakcji agentów poprzez pomiar czasu i logowanie, tworząc podstawy dla śledzenia i monitorowania.
2. **Automatycznie oceniać odpowiedzi agenta** za pomocą agenta oceniającego, który ocenia kompletność, dokładność i przydatność.
3. **Zarządzać kosztami** poprzez optymalizację promptów, dobór modeli, buforowanie i budżety tokenów.

Te wzorce produkcyjne pomagają zapewnić, że twoi agenci AI są niezawodni, mierzalni i opłacalni w skali.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Wyłączenie odpowiedzialności:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczeniowej opartej na sztucznej inteligencji Co-op Translator (https://github.com/Azure/co-op-translator). Chociaż dokładamy starań o dokładność, prosimy pamiętać, że tłumaczenia automatyczne mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku oryginalnym należy uznać za dokument wiążący. W przypadku informacji o krytycznym znaczeniu zalecane jest skorzystanie z usług profesjonalnego tłumacza. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
